# CT simulator validation tests (Google Colab)

This notebook runs **validation** comparisons against independent physics or geometric references. It covers monochromatic attenuation, projection geometry and a rotating off-axis sphere, multi-material additivity, detector-effect statistics, and C++ voxelized volume. The tests report measured errors and review statuses; they do not certify agreement with a clinical scanner.

The aluminum coefficients are entered by hand from NIST's tabulated total mass-attenuation data. NIST describes the relationship between its XCOM and XAAMDI data sets in its [database note](https://www.nist.gov/pml/note-nist-x-ray-attenuation-databases). Include the reference links and exact values alongside reported results. The project uses its gVXR material model for the simulation; the reference calculation below does not call xraylib.


## 1. Upload the project files

Upload the current `ct_core.py`, `ct_variations.py`, and `ct_voxel.cpp` from the project when prompted.


In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content')
required = ['ct_core.py', 'ct_variations.py', 'ct_voxel.cpp']
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    from google.colab import files
    files.upload()
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Upload these project files: ' + ', '.join(missing))


## 2. Install Colab dependencies

Run once per runtime. The system packages include the XCB/OpenGL libraries required by gVXR. `ct_core.py` selects the best available renderer before creating its OpenGL/EGL context.


In [ ]:
import subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'build-essential', 'libgomp1', 'libxcb-res0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxcb-icccm4', 'libxcb-image0',
                'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-shape0', 'libxcb-randr0', 'libxkbcommon-x11-0',
                'libxrandr2', 'libxinerama1', 'libxcursor1', 'libxi6', 'libxxf86vm1',
                'libgl1', 'libegl1', 'libopengl0', 'libglu1-mesa', 'libgl1-mesa-dri', 'libegl-mesa0', 'libglx-mesa0',
                'libxcb-composite0', 'libxcb-cursor0', 'libxcb-dri2-0', 'libxcb-ewmh2',
                'libxcb-dri3-0', 'libxcb-glx0', 'libxcb-present0', 'libxcb-sync1', 'libxcb-xfixes0', 'libxcb-xinput0',
                'libxcb-util1', 'libxcb-xv0', 'libxcb-damage0', 'libxcb-dpms0', 'libxcb-record0', 'libxcb-screensaver0',
                'libxcb-shm0', 'libxcb-xtest0', 'libxcb-xf86dri0', 'libxcb-xvmc0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gvxr', 'numpy', 'trimesh', 'scipy', 'scikit-image', 'tifffile', 'h5py', 'matplotlib', 'pandas', 'pybind11'], check=True)


## 3. Import code and set up the phantom

The main tests use a 10 mm aluminium sphere, a cone-beam geometry with magnification 1.25, and an odd detector width so a pixel lies on the central ray.


In [ ]:
import importlib, json, math, platform, sys, sysconfig, tempfile
import numpy as np
import pandas as pd
import trimesh
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT_DIR.resolve()))
import ct_core as core
import ct_variations as variations

RADIUS_MM = 5.0
DIAMETER_MM = 2.0 * RADIUS_MM
AL_DENSITY = 2.70
FE_DENSITY = 7.874
SOD_CM, ODD_CM = 40.0, 10.0
MAG = (SOD_CM + ODD_CM) / SOD_CM
NX, NY, PIXEL_MM = 401, 3, 0.1
PHOTONS = 100000

sphere_path = Path(tempfile.gettempdir()) / 'ct_validation_al_sphere.stl'
trimesh.creation.icosphere(subdivisions=5, radius=RADIUS_MM).export(sphere_path)
al_sphere = core.ObjectCfg('al_sphere', str(sphere_path), 'Element', 'Al', AL_DENSITY)

def make_cfg(objects, energy_kev=60.0, nx=NX, ny=NY, pixel_mm=PIXEL_MM,
             n_angles=1, total_deg=360.0):
    return core.ScanCfg(objects=objects, stl_unit='mm', center_each=True, beam='Cone',
                        source_x_cm=SOD_CM, detector_x_cm=ODD_CM, energy_kev=energy_kev,
                        photons=PHOTONS, nx=nx, ny=ny, pixel_mm=pixel_mm,
                        n_angles=n_angles, total_deg=total_deg, noise=False, spectrum=None)

def project_one(cfg):
    flat = core.on_gl_thread(core.build_scene, cfg)
    raw = core.on_gl_thread(lambda: np.asarray(core._gx().computeXRayImage(), dtype=np.float32))
    attenuation = -np.log(np.maximum(raw, 1e-30) / np.maximum(flat, 1e-30))
    return raw, flat, attenuation

def status(condition):
    return 'PASS' if bool(condition) else 'REVIEW'

renderer = core.on_gl_thread(lambda: (core._gx(), core.renderer_info())[1])
print('OpenGL renderer:', renderer)
print('Renderer selection:', core.renderer_selection_info())


## Validation 1 — Monochromatic attenuation against NIST

The center ray crosses the sphere’s 10 mm diameter. Convert NIST’s mass attenuation coefficient with `μ = (μ/ρ)ρ`, then compare `μL` with simulated `−ln(I/I₀)`. NIST total mass attenuation values (cm²/g): Al at 30 keV = 1.128; 60 keV = 0.2778; 100 keV = 0.1704. A 10% threshold below is an initial review threshold; report the measured errors rather than treating it as a universal accuracy standard.

Reference: [NIST Aluminum attenuation table](https://physics.nist.gov/PhysRefData/XrayMassCoef/ElemTab/z13.html).


In [ ]:
# Manually transcribed NIST total mass attenuation coefficients, in cm^2/g.
NIST_AL_MU_OVER_RHO = {30.0: 1.128, 60.0: 0.2778, 100.0: 0.1704}
physics_results = []
for energy_kev, mass_mu_cm2_g in NIST_AL_MU_OVER_RHO.items():
    cfg = make_cfg([al_sphere], energy_kev=energy_kev)
    raw, flat, attenuation = project_one(cfg)
    measured = float(attenuation[NY // 2, NX // 2])
    path_cm = DIAMETER_MM / 10.0
    mu_cm_inv = mass_mu_cm2_g * AL_DENSITY
    expected = mu_cm_inv * path_cm
    error_pct = 100.0 * (measured - expected) / expected
    physics_results.append({
        'energy_keV': energy_kev,
        'NIST_mu_over_rho_cm2_g': mass_mu_cm2_g,
        'expected_muL': expected,
        'measured_minus_ln_I_over_I0': measured,
        'error_percent': error_pct,
        'status_within_10pct': status(abs(error_pct) <= 10.0),
    })
physics_df = pd.DataFrame(physics_results)
display(physics_df)


## Validation 2 — Geometry and off-axis sinusoid

The expected projected sphere width is `diameter × magnification / pixel_mm`. Next, rotate a sphere whose center starts 8 mm off-axis. Fit its projection centroid to `A cos(θ) + B sin(θ) + C`; compare fitted amplitude with 8 mm and phase with the expected extremum at the starting angle (0° or 180° depending on detector-axis sign).


In [ ]:
# Measure the 60 keV projection width at a low threshold.
cfg_width = make_cfg([al_sphere], energy_kev=60.0)
_, _, att_width = project_one(cfg_width)
profile = att_width[NY // 2]
threshold = max(float(profile.max()) * 0.01, 1e-6)
support = np.flatnonzero(profile > threshold)
measured_width_px = int(support[-1] - support[0] + 1)
expected_width_px = DIAMETER_MM * MAG / PIXEL_MM
width_error_px = measured_width_px - expected_width_px
print({'measured_width_px': measured_width_px, 'expected_width_px': expected_width_px,
       'error_px': width_error_px,
       'status_within_3px': status(abs(width_error_px) <= 3.0)})

# Track a sphere offset along world y through one rotation.
OFF_AXIS_MM = 8.0
N_VIEWS = 180
off_axis_sphere = core.ObjectCfg('off_axis_sphere', str(sphere_path), 'Element', 'Al', AL_DENSITY,
                                 offset_mm=(0.0, OFF_AXIS_MM, 0.0))
cfg_orbit = make_cfg([off_axis_sphere], energy_kev=60.0, n_angles=N_VIEWS, total_deg=360.0)
flat_orbit = core.on_gl_thread(core.build_scene, cfg_orbit)
raw_stack = core.on_gl_thread(core.acquire, N_VIEWS, 360.0 / N_VIEWS)
att_stack = -np.log(np.maximum(raw_stack, 1e-30) / np.maximum(flat_orbit[None, :, :], 1e-30))
cols = np.arange(NX, dtype=float)
centroids_px = []
for view in att_stack[:, NY // 2, :]:
    weights = np.maximum(view, 0.0)
    centroids_px.append(float(np.sum(cols * weights) / max(float(np.sum(weights)), 1e-20)))
angles_deg = np.arange(N_VIEWS) * (360.0 / N_VIEWS)
centroids_obj_mm = (np.asarray(centroids_px) - (NX - 1) / 2) * PIXEL_MM / MAG
basis = np.column_stack([np.cos(np.deg2rad(angles_deg)),
                         np.sin(np.deg2rad(angles_deg)), np.ones(N_VIEWS)])
coef, *_ = np.linalg.lstsq(basis, centroids_obj_mm, rcond=None)
amplitude_mm = float(np.hypot(coef[0], coef[1]))
phase_deg = float(np.degrees(np.arctan2(coef[1], coef[0])) % 360.0)
phase_error_deg = min(phase_deg, abs(phase_deg - 180.0), 360.0 - phase_deg)
fit = basis @ coef
fit_rmse_mm = float(np.sqrt(np.mean((centroids_obj_mm - fit) ** 2)))
print({'expected_amplitude_mm': OFF_AXIS_MM, 'measured_amplitude_mm': amplitude_mm,
       'amplitude_error_percent': 100.0 * (amplitude_mm - OFF_AXIS_MM) / OFF_AXIS_MM,
       'fitted_phase_deg': phase_deg, 'phase_error_to_0_or_180_deg': phase_error_deg,
       'fit_rmse_mm': fit_rmse_mm,
       'amplitude_status_within_5pct': status(abs(amplitude_mm - OFF_AXIS_MM) / OFF_AXIS_MM <= 0.05),
       'phase_status_within_5deg': status(phase_error_deg <= 5.0)})
plt.figure(figsize=(9, 4))
plt.plot(angles_deg, centroids_obj_mm, '.', ms=3, label='Measured centroid')
plt.plot(angles_deg, fit, '-', label='Sinusoid fit')
plt.xlabel('View angle (degrees)'); plt.ylabel('Object-space centroid (mm)')
plt.grid(alpha=0.25); plt.legend(); plt.show()


## Validation 3 — Multi-material attenuation additivity

Put 6 mm aluminum and 6 mm iron slabs end-to-end along the central ray. Compare the measured combined attenuation with `μAl LAl + μFe LFe` and with the sum of separate slab projections. At 60 keV, the NIST mass attenuation values are Al = 0.2778 cm²/g and Fe = 1.205 cm²/g. Densities are set to 2.70 and 7.874 g/cm³ respectively.

Reference: [NIST Aluminum table](https://physics.nist.gov/PhysRefData/XrayMassCoef/ElemTab/z13.html), [NIST Iron table](https://physics.nist.gov/PhysRefData/XrayMassCoef/ElemTab/z26.html).


In [ ]:
AL_MU_RHO_60 = 0.2778
FE_MU_RHO_60 = 1.205
SLAB_MM = 6.0
al_slab_path = Path(tempfile.gettempdir()) / 'validation_al_slab.stl'
fe_slab_path = Path(tempfile.gettempdir()) / 'validation_fe_slab.stl'
trimesh.creation.box(extents=(SLAB_MM, 20.0, 20.0)).export(al_slab_path)
trimesh.creation.box(extents=(SLAB_MM, 20.0, 20.0)).export(fe_slab_path)
al_slab = core.ObjectCfg('al_slab', str(al_slab_path), 'Element', 'Al', AL_DENSITY,
                         offset_mm=(-SLAB_MM / 2, 0.0, 0.0))
fe_slab = core.ObjectCfg('fe_slab', str(fe_slab_path), 'Element', 'Fe', FE_DENSITY,
                         offset_mm=(SLAB_MM / 2, 0.0, 0.0))

def measured_center(objects):
    cfg = make_cfg(objects, energy_kev=60.0, nx=101, ny=3, pixel_mm=0.2)
    _, _, att = project_one(cfg)
    return float(att[1, 50])

al_measured = measured_center([al_slab])
fe_measured = measured_center([fe_slab])
combined_measured = measured_center([al_slab, fe_slab])
expected_nist = (AL_MU_RHO_60 * AL_DENSITY + FE_MU_RHO_60 * FE_DENSITY) * (SLAB_MM / 10.0)
additivity_error_pct = 100.0 * (combined_measured - al_measured - fe_measured) / expected_nist
nist_error_pct = 100.0 * (combined_measured - expected_nist) / expected_nist
multi_results = {
    'al_only_measured': al_measured,
    'fe_only_measured': fe_measured,
    'combined_measured': combined_measured,
    'sum_of_separate_measurements': al_measured + fe_measured,
    'NIST_expected_muL_sum': expected_nist,
    'additivity_error_percent': additivity_error_pct,
    'NIST_error_percent': nist_error_pct,
    'additivity_status_within_5pct': status(abs(additivity_error_pct) <= 5.0),
    'NIST_status_within_10pct': status(abs(nist_error_pct) <= 10.0),
}
print(multi_results)


## Validation 4 — Detector effects

Use synthetic signals with known statistics to test the project’s effect functions. For Poisson noise, variance/mean should approach 1 and SNR should scale approximately as √photons. For blur, estimate Gaussian σ from the 10–90% edge-spread width (`width ≈ 2.563σ`). For gain, measure open-field pixel variation against `gain_sigma`; for flat mismatch, measure the fixed attenuation pattern that produces ring-artifact input.


In [ ]:
# Measure Poisson noise at several photon counts.
stat_ny, stat_nx, n_scans = 80, 80, 20
flat_uniform = np.ones((stat_ny, stat_nx), dtype=np.float32)
raw_uniform = np.ones((n_scans, stat_ny, stat_nx), dtype=np.float32)
poisson_results = []
for nphot in (500, 2000, 8000, 32000):
    fx = variations.EffectsCfg(poisson=True, photons=nphot, flat_frames=1, seed=1234)
    _, counts, _ = variations.apply_effects(raw_uniform, flat_uniform, nphot, fx)
    sample = counts.ravel().astype(float)
    mean = float(sample.mean()); variance = float(sample.var(ddof=1)); snr = mean / np.sqrt(variance)
    poisson_results.append({'photons': nphot, 'mean_counts': mean, 'variance_counts2': variance,
                            'variance_over_mean': variance / mean, 'SNR': snr,
                            'status_variance_within_5pct': status(abs(variance / mean - 1.0) <= 0.05)})
noise_df = pd.DataFrame(poisson_results)
display(noise_df)
snr_slope, _ = np.polyfit(np.log(noise_df['photons']), np.log(noise_df['SNR']), 1)
print({'SNR_log_log_slope': float(snr_slope), 'expected_slope': 0.5,
       'status_within_0.05': status(abs(snr_slope - 0.5) <= 0.05)})
plt.figure(figsize=(6, 4))
plt.loglog(noise_df['photons'], noise_df['SNR'], 'o-', label='Measured')
plt.loglog(noise_df['photons'], noise_df['SNR'].iloc[0] * np.sqrt(noise_df['photons'] / noise_df['photons'].iloc[0]), '--', label='√photons reference')
plt.xlabel('Photons per pixel'); plt.ylabel('SNR'); plt.grid(True, which='both', alpha=.25); plt.legend(); plt.show()

# Blur: recover sigma from a sharp vertical edge.
SIGMA_TRUE_PX = 2.0
edge_ny, edge_nx, edge_col = 201, 401, 200
edge_flat = np.ones((edge_ny, edge_nx), dtype=np.float32)
edge_raw = np.ones_like(edge_flat); edge_raw[:, :edge_col] = 0.0
fx_blur = variations.EffectsCfg(psf_sigma_px=SIGMA_TRUE_PX, seed=1)
_, blurred_counts, _ = variations.apply_effects(edge_raw, edge_flat, 10000, fx_blur)
edge_profile = blurred_counts.mean(axis=0) / 10000.0

def edge_crossing(values, level):
    indices = np.flatnonzero(values >= level)
    j = int(indices[0])
    y0, y1 = values[j - 1], values[j]
    return (j - 1) + (level - y0) / (y1 - y0)

x10, x90 = edge_crossing(edge_profile, 0.1), edge_crossing(edge_profile, 0.9)
sigma_recovered_px = float((x90 - x10) / 2.563103)
blur_result = {'sigma_config_px': SIGMA_TRUE_PX, 'edge_10_90_width_px': float(x90-x10),
               'sigma_recovered_px': sigma_recovered_px,
               'status_within_10pct': status(abs(sigma_recovered_px-SIGMA_TRUE_PX)/SIGMA_TRUE_PX <= .10)}
print(blur_result)
plt.figure(figsize=(6, 4)); plt.plot(edge_profile)
plt.xlim(edge_col-12, edge_col+12); plt.xlabel('Detector column (pixels)')
plt.ylabel('Normalized counts'); plt.title('Blurred edge-spread profile'); plt.grid(alpha=.25); plt.show()

# Measure pixel gain and scan-to-flat mismatch.
GAIN_SIGMA = 0.05
gain_fx = variations.EffectsCfg(gain_sigma=GAIN_SIGMA, seed=2026)
_, _, flat_gain = variations.apply_effects(raw_uniform[0], flat_uniform, 10000, gain_fx)
gain_cv = float(flat_gain.std(ddof=1) / flat_gain.mean())
MISMATCH_SIGMA = 0.03
mismatch_fx = variations.EffectsCfg(flat_mismatch_sigma=MISMATCH_SIGMA, seed=2027)
atten_mismatch, _, _ = variations.apply_effects(raw_uniform[0], flat_uniform, 10000, mismatch_fx)
mismatch_sd = float(atten_mismatch.std(ddof=1))
detector_result = {
    'gain_sigma_config': GAIN_SIGMA, 'flat_pixel_CV_measured': gain_cv,
    'gain_status_within_10pct': status(abs(gain_cv-GAIN_SIGMA)/GAIN_SIGMA <= .10),
    'flat_mismatch_sigma_config': MISMATCH_SIGMA, 'fixed_attenuation_SD_measured': mismatch_sd,
    'ring_pattern_status_within_15pct': status(abs(mismatch_sd-MISMATCH_SIGMA)/MISMATCH_SIGMA <= .15),
}
print(detector_result)


## Validation 5 — Voxelized volume and pitch convergence

Build the project’s OpenMP C++ voxelizer, then compare voxel-count volume for a watertight sphere and cube with the exact formula and `trimesh`’s mesh volume. The sphere’s absolute error should generally reduce at finer pitch; voxel-grid alignment can make individual steps non-monotonic.


In [ ]:
suffix = sysconfig.get_config_var('EXT_SUFFIX') or '.so'
voxel_module = PROJECT_DIR / ('ct_voxel' + suffix)
compile_cmd = ['c++', '-O3', '-std=c++17', '-shared', '-fPIC', '-fopenmp',
               '-I' + __import__('pybind11').get_include(),
               '-I' + sysconfig.get_path('include'), str(PROJECT_DIR / 'ct_voxel.cpp'),
               '-o', str(voxel_module)]
subprocess.check_call(compile_cmd, cwd=PROJECT_DIR)
importlib.invalidate_caches()
import ct_voxel

def voxelized_volume(mesh, pitch_mm):
    bounds = mesh.bounds
    origin = np.floor(bounds[0] / pitch_mm) * pitch_mm - pitch_mm
    upper = np.ceil(bounds[1] / pitch_mm) * pitch_mm + pitch_mm
    shape = np.maximum(np.ceil((upper - origin) / pitch_mm).astype(int), 1)
    mask = ct_voxel.voxelize(np.asarray(mesh.vertices, dtype=np.float64),
                             np.asarray(mesh.faces, dtype=np.int64), tuple(map(float, origin)),
                             float(pitch_mm), tuple(map(int, shape)))
    return int(np.count_nonzero(mask)) * pitch_mm**3

sphere_mesh = trimesh.creation.icosphere(subdivisions=5, radius=RADIUS_MM)
cube_mesh = trimesh.creation.box(extents=(10.0, 10.0, 10.0))
exact_sphere = 4.0/3.0 * np.pi * RADIUS_MM**3
exact_cube = 1000.0
voxel_results = []
for name, mesh, exact in [('sphere', sphere_mesh, exact_sphere), ('cube', cube_mesh, exact_cube)]:
    for pitch in (1.0, 0.5, 0.25):
        voxel_volume_mm3 = voxelized_volume(mesh, pitch)
        voxel_results.append({
            'shape': name, 'pitch_mm': pitch, 'voxel_volume_mm3': voxel_volume_mm3,
            'exact_volume_mm3': exact, 'trimesh_volume_mm3': float(mesh.volume),
            'error_vs_exact_percent': 100.0 * (voxel_volume_mm3 - exact) / exact,
            'error_vs_trimesh_percent': 100.0 * (voxel_volume_mm3 - float(mesh.volume)) / float(mesh.volume),
        })
voxel_df = pd.DataFrame(voxel_results)
display(voxel_df)


## Save the validation report

The JSON and CSV preserve measured values for discussion. A `PASS` threshold is an initial engineering screen; inspect and explain any `REVIEW` result. Record the renderer because it affects runtime, while these validation comparisons assess numerical agreement.


In [ ]:
report = {
    'suite': 'CT simulator independent validation',
    'references': {
        'NIST_aluminum': 'https://physics.nist.gov/PhysRefData/XrayMassCoef/ElemTab/z13.html',
        'NIST_iron': 'https://physics.nist.gov/PhysRefData/XrayMassCoef/ElemTab/z26.html',
    },
    'aluminum_monoenergetic': physics_results,
    'geometry': {
        'sphere_width_measured_px': measured_width_px,
        'sphere_width_expected_px': float(expected_width_px),
        'sphere_width_status': status(abs(width_error_px) <= 3.0),
        'offaxis_amplitude_expected_mm': OFF_AXIS_MM,
        'offaxis_amplitude_measured_mm': amplitude_mm,
        'offaxis_phase_deg': phase_deg,
        'offaxis_phase_error_deg': phase_error_deg,
        'offaxis_fit_rmse_mm': fit_rmse_mm,
    },
    'multi_material': multi_results,
    'detector_effects': {
        'poisson': poisson_results, 'snr_log_log_slope': float(snr_slope),
        'blur': blur_result, 'gain_and_ring_pattern': detector_result,
    },
    'voxelizer': voxel_results,
    'renderer': core.renderer_info(),
    'renderer_selection': core.renderer_selection_info(),
    'runtime': {'python': sys.version.split()[0], 'platform': platform.platform()},
}
report_path = Path('ct_simulator_validation_report.json')
report_path.write_text(json.dumps(report, indent=2))
physics_df.to_csv('validation_aluminum_physics.csv', index=False)
voxel_df.to_csv('validation_voxel_volume.csv', index=False)
print('Saved:', report_path.resolve())
try:
    from google.colab import files
    files.download(str(report_path))
except ImportError:
    print(report_path.read_text())
